In [23]:
import pandas as pd
import numpy as np
import random
from collections import Counter

In [24]:
POPULASI = 50
GENERASI = 100

PROBABILITAS_CROSSOVER = 0.7
PROBABILITAS_MUTASI = 0.4
BIAS_CROSSOVER = 0.85
BIAS_MUTASI = 0.85

VIOLATION_COST = 100
TOURNAMENT_SIZE = 10

In [25]:
guru_df = pd.read_csv('../dataset/guru.csv')
kelas_df = pd.read_csv('../dataset/kelas.csv')
mapel_df = pd.read_csv('../dataset/mapel.csv')
relasi_guru_mapel_df = pd.read_csv('../dataset/relasi_guru_mapel.csv')
slot_df = pd.read_csv('../dataset/slot.csv')

In [26]:
join = (
    relasi_guru_mapel_df
    .merge(guru_df, on="guru_id", how="left")
    .merge(mapel_df, on="mapel_id", how="left")
)

In [27]:
join["total"] = join.groupby("guru_id")["durasi"].transform("sum")

In [28]:
mapping_hari = {
    "Senin" : 1,
    "Selasa" : 2,
    "Rabu" : 3,
    "Kamis" : 4,
    "Jumat" : 5
}

In [29]:
relasi = join[[
    "guru_id", "mapel_id", "jam_per_minggu", "tingkatan", 
    "durasi", "total", "MGMP"
]].copy()

In [31]:
relasi["MGMP"] = relasi["MGMP"]. map(mapping_hari)

In [32]:
mapel_jam = dict(zip(mapel_df["mapel_id"], mapel_df["jam_per_minggu"]))

In [33]:
slot_per_hari = {
    1:8,
    2:8,
    3:8,
    4:7,
    5:5
}

In [34]:
total_kelas = kelas_df["kelas_id"].count()

In [35]:
batas_siang = {0: 5, 1: 5, 2: 4, 3: 5, 4: 4}
batas_mgmp = {0: 2, 1: 2, 2: 2, 3: 2, 4: 1}

In [36]:
guru_by_mapel = (
    relasi
    .groupby(["mapel_id", "tingkatan"])["guru_id"]
    .apply(list)
    .to_dict()
)

In [37]:
MAPEL_LIST = list(range(1, 14))

# SELEKSI TURNAMEN

In [ ]:
def tournament_selection(populasi, fitness):
    kandidat = random.sample(list(zip(populasi, fitness)), TOURNAMENT_SIZE)
    kandidat.sort(key=lambda x: x[1])
    return kandidat[0][0]

# MULTI-POINT CROSSOVER

In [38]:
def crossover(p1, p2, mask1):
    c1 = np.array(p1, dtype=object)
    c2 = np.array(p2, dtype=object)

    for k in range(len(c1)):
        for h in range(len(c1[k]) - 1):

            if random.random() > PROBABILITAS_CROSSOVER:
                continue

            slot_len = len(c1[k][h])
            mask_flat = np.array(mask1[k][h])

            zero_id = np.where(mask_flat == 0)[0]
            if len(zero_id) > 0 and random.random() < BIAS_CROSSOVER:
                point = np.random.choice(zero_id, size=random.randint(1, len(zero_id)), replace=False)

            else:
                point = np.random.choice(slot_len, size=random.randint(1, slot_len), replace=False)

            for id in point:
                c1[k][h][id], c2[k][h][id] = c2[k][h][id], c1[k][h][id]

    return c1.tolist(), c2.tolist()

# SCRAMBLED MUTATION

In [ ]:
def mutasi(individu, mask):
    for k in range(len(individu)):
        for h in range(len(individu[k]) - 1 ):

            if random.random() > PROBABILITAS_MUTASI:
                continue
                
            hari = individu[k][h]
            hari_mask = np.array(mask[k][h])

            zero_id = np.where(hari_mask == 0)[0]

            if len(zero_id) > 1 and random.random() < BIAS_MUTASI:
                id = zero_id
            else:
                id = np.random.choice(len(hari), size=random.randint(2, len(hari)), replace=False)

            value = [hari[i] for i in id]
            random.shuffle(value)

            for i,v in zip(id, value):
                hari[i] = v
    
    return individu